# 1. Data exploration
## Phase 4 data files
Logging in to Hugging Face, mounting Drive and listing the Phase 4 judgment prediction files in the dataset.


In [2]:
import os
from huggingface_hub import login, get_token, list_repo_files
from google.colab import drive

login()
# Passing the token on so !python scripts can also download the data
os.environ["HF_TOKEN"] = get_token()

drive.mount('/content/drive')

# Listing every file in the dataset that belongs to Phase 4
all_files = list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset")
phase4_files = [f for f in all_files if "Phase_4" in f]

for f in phase4_files:
    print(f)


Mounted at /content/drive
Phase_4_CJPE/Test/test.json
Phase_4_CJPE/Train/dev.json
Phase_4_CJPE/Train/train.json


## Phase 4 files and format
Downloading the train, dev and test files and checking how each case is stored.


In [3]:
# Downloading all the Phase 4 files and looking at how they are stored
import json
from huggingface_hub import hf_hub_download

paths = {}
for f in phase4_files:
    name = f.split("/")[-1]
    paths[name] = hf_hub_download("Exploration-Lab/CS-781-Capstone", f, repo_type="dataset")

data = {}
for name in ["train.json", "dev.json", "test.json"]:
    with open(paths[name]) as f:
        data[name] = json.load(f)

    print()
    print(name, type(data[name]), len(data[name]))
    print(str(data[name])[:600])


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            


train.json <class 'list'> 37387
[{'id': '2020_1', 'text': ' Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of the High Court 1 dismissing S.T. No.192 of 2014 and other companynected matters and thereby affirming the decision of the Single Judge of the High Court passed on 12.03.2014 in Writ Petition No.20650  W  of 2013 which in turn had found Sections 8, 10, 11 and 12 of the West Bengal Madrasah Service Commission Act, 2008  the Commission Act, for short  to be ultra vires. 1 The High Court of Judicature at Calcutta Civil Appeal No.5808 of 2017 SK. MD. Ra

dev.json <class 'list'> 1988
[{'id': '1989_75', 'text': 'civil appellate jurisdiction civil appeal number 1709\nof\n1988.\nfrom the judgment and order dated 9.10.1987 of t\nhe\ncentral administrative tribunal chandigarh in appln. t.\n--\n1055 of 1986.\nwith\n slp  civil  number 6998 of 1988 \n\nc. mahajan mrs. indu goswami c.v. subba rao\np.\nparmeshwaran for the

## Cases and labels
Counting the cases in each split, how many have label 0 and label 1, and how many cases are fully lowercase.


In [3]:
from collections import Counter

train = data["train.json"]
dev = data["dev.json"]
test = data["test.json"]

print("train keys:", list(train[0].keys()))
print("dev keys:", list(dev[0].keys()))
print("test keys:", list(test[0].keys()))

# Finding the name of the label field (the key that is not id or text)
label_key = [k for k in train[0] if k not in ("id", "text")][0]
print("label field:", label_key)

for name, cases in [("train", train), ("dev", dev)]:
    label_counts = Counter(case[label_key] for case in cases)
    print()
    print(name, "-", len(cases), "cases")
    print(label_counts.most_common())

print()
print("test -", len(test), "cases")
print("unique test ids:", len(set(case["id"] for case in test)))

# Counting cases that are fully lowercase (like dev and test) in each split
for name, cases in [("train", train), ("dev", dev), ("test", test)]:
    lowercase = sum(case["text"] == case["text"].lower() for case in cases)
    print(name, "lowercase cases:", lowercase, "of", len(cases))


train keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
dev keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
test keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
label field: label

train - 37387 cases
[(0, 22067), (1, 15320)]

dev - 1988 cases
[(0, 994), (1, 994)]

test - 1517 cases
unique test ids: 1517
train lowercase cases: 0 of 37387
dev lowercase cases: 1988 of 1988
test lowercase cases: 1500 of 1517


## Extra keys in the data
Checking what the expert keys hold, and that the test label key is empty.


In [4]:
# Checking that the test labels are empty (they are private, so they are never used)
empty_test_labels = sum(case["label"] in (None, "", -1) for case in test)
print("test cases with an empty label:", empty_test_labels, "of", len(test))

# Counting how many cases have something in each expert key
for name, cases in [("train", train), ("dev", dev), ("test", test)]:
    for key in ["expert_1", "expert_2", "expert_3", "expert_4", "expert_5"]:
        filled = sum(bool(case[key]) for case in cases)
        print(name, key, "filled in:", filled)

# Looking at one filled expert value in dev, if there is one
for case in dev:
    if case["expert_1"]:
        print()
        print("example expert_1:", str(case["expert_1"])[:500])
        break


test cases with an empty label: 0 of 1517
train expert_1 filled in: 0
train expert_2 filled in: 0
train expert_3 filled in: 0
train expert_4 filled in: 0
train expert_5 filled in: 0
dev expert_1 filled in: 0
dev expert_2 filled in: 0
dev expert_3 filled in: 0
dev expert_4 filled in: 0
dev expert_5 filled in: 0
test expert_1 filled in: 0
test expert_2 filled in: 0
test expert_3 filled in: 0
test expert_4 filled in: 0
test expert_5 filled in: 0


## Case lengths in tokens
Checking case lengths with the Phase-1 tokenizer, after lowercasing, and looking at the start and end of one case.


In [5]:
# Counting tokens per case with the Phase-1 tokenizer, on lowercased text
import random
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("/content/drive/MyDrive/CS781/data_p1/tokenizer.json")

# Using a random 3000 train cases, tokenizing all 37k on CPU takes too long
train_sample = random.Random(1337).sample(train, 3000)

for name, cases in [("train (3000 sample)", train_sample), ("dev", dev), ("test", test)]:
    texts = [case["text"].lower() for case in cases]
    lengths = sorted(len(enc.ids) for enc in tokenizer.encode_batch(texts))

    print()
    print(name)
    print("tokens per case - mean:", sum(lengths) // len(lengths),
          "| median:", lengths[len(lengths) // 2],
          "| 95%:", lengths[int(0.95 * len(lengths))],
          "| max:", lengths[-1])
    print("cases longer than 512 tokens:", sum(l > 512 for l in lengths),
          "| longer than 2048:", sum(l > 2048 for l in lengths),
          "| longer than 4096:", sum(l > 4096 for l in lengths))

# Looking at the start and end of one dev case
print()
print("dev label:", dev[0]["label"])
print("START:", dev[0]["text"][:600])
print()
print("END:", dev[0]["text"][-600:])



train (3000 sample)
tokens per case - mean: 4332 | median: 2795 | 95%: 12017 | max: 477406
cases longer than 512 tokens: 2703 | longer than 2048: 1865 | longer than 4096: 970

dev
tokens per case - mean: 5049 | median: 3809 | 95%: 12596 | max: 60474
cases longer than 512 tokens: 1972 | longer than 2048: 1642 | longer than 4096: 906

test
tokens per case - mean: 5176 | median: 3957 | 95%: 13671 | max: 56370
cases longer than 512 tokens: 1500 | longer than 2048: 1265 | longer than 4096: 720

dev label: 0
START: civil appellate jurisdiction civil appeal number 1709
of
1988.
from the judgment and order dated 9.10.1987 of t
he
central administrative tribunal chandigarh in appln. t.
--
1055 of 1986.
with
 slp  civil  number 6998 of 1988 

c. mahajan mrs. indu goswami c.v. subba rao
p.
parmeshwaran for the appellant in c.a. number 1709 of 1988.

k.d. namboodary for the petitioner in slp  civil  n
o. 6998 of 1988.

m. ashri and mahabir singh for the respondents. the judgment of the companyrt 

## Endings of some train cases
Reading the last part of a few train cases with their labels, to see if the decision is still in the text.


In [7]:
# Printing the last 300 characters of 6 random train cases with their labels
for case in random.Random(7).sample(train, 6):
    print("label:", case["label"])
    print(case["text"][-300:])
    print("-" * 80)


label: 1
nyclusion. Therefore,. I think it right to give the party, who appears to be unable to represent his own case, an opportunity to present his grievance through, his friend. That friend, judging by the numbere prepared and put in, seems to be familiar with law, although quacks can prove fatal friends.
--------------------------------------------------------------------------------
label: 0
htly held that the Magistrate is number bound to take companynizance of the offences indicated in the police report. That being so, the ultimate directions of the High Court materially differ from its earlier companyclusions. The Magistrate has to companysider material at the time of framing charge.
--------------------------------------------------------------------------------
label: 0
executed by the appellants and the right of the respondent-landlord to seek the relief as prayed in the present suit. The learned Judge of the High Court went into evidence and referring to the statement of Su

# 2. Data preparation
## Making the chunks
Running data_prep.py for cleaning the cases and keeping the last 2048 tokens of each case as 4 chunks of 512 tokens. Training on all 37k train cases would take about 2 hours per epoch on the free Colab T4, so a random balanced subset of 6000 train cases (3000 for each label) is used.




In [8]:
import os
from huggingface_hub import login, get_token
from google.colab import drive

login()
os.environ["HF_TOKEN"] = get_token()
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase4/data_prep.py \
    --out_dir /content/drive/MyDrive/CS781/data_p4 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full train label counts: {0: 22067, 1: 15320}
Tokenizing and making chunks ...
  train: 6000 cases, 3.41 chunks per case on average
  dev: 1988 cases, 3.84 chunks per case on average
  test: 1517 cases, 3.83 chunks per case on average

Saved to /content/drive/MyDrive/CS781/data_p4
Train subset label counts: {1: 3000, 0: 3000}


## Testing the model
Checking that the model works on two small fake cases before training.



In [11]:
# Checking the model on two small fake cases
import sys
import torch
sys.path.append("/content/drive/MyDrive/CS781/phase4")

from model import ModelConfig, build_cjpe_model, count_parameters

# Using the same size as the Phase-1 decoder
config = ModelConfig(arch="decoder", d_model=768, n_layer=12, n_head=12, ffn_hidden=2048)
model = build_cjpe_model(config)
print("Model parameters:", count_parameters(model) / 1e6, "M")

# Making two fake cases with 4 chunks of 64 random tokens
token_ids = torch.randint(0, config.vocab_size, (2, 4, 64))
token_mask = torch.ones(2, 4, 64)

# Second case has only 2 real chunks, the last 2 are padding
token_mask[1, 2:] = 0
labels = torch.tensor([0, 1])

logits, loss = model(token_ids, token_mask, labels)
print("logits shape:", logits.shape)
print("loss:", loss.item())


Model parameters: 111.63213 M
logits shape: torch.Size([2, 2])
loss: 0.7467246055603027


## Testing methods.py
Checking the four fine-tuning methods (full fine-tuning, LoRMA, BitFit and partial fine-tuning) and how many weights each one trains.


In [12]:
# Building a small model with each method and counting trainable weights
import sys
sys.path.append("/content/drive/MyDrive/CS781/phase4")

from model import ModelConfig, build_cjpe_model
from methods import setup_method, count_trainable

method_names = {
    "full": "Full fine-tuning",
    "lorma": "LoRMA",
    "bitfit": "BitFit",
    "partial": "Partial fine-tuning (last 4 blocks)",
}

config = ModelConfig(arch="convmix", n_layer=6)
for method, name in method_names.items():
    model = build_cjpe_model(config)
    setup_method(model, method)
    trainable, total = count_trainable(model)
    print(f"{name}: training {trainable:,} of {total:,} weights ({100 * trainable / total:.2f}%)")


Full fine-tuning: training 66,820,866 of 66,820,866 weights (100.00%)
LoRMA: training 2,348,034 of 67,066,626 weights (3.50%)
BitFit: training 2,149,890 of 66,865,410 weights (3.22%)
Partial fine-tuning (last 4 blocks): training 29,253,378 of 66,820,866 weights (43.78%)


## Full fine-tuning: decoder
Training all the weights of the Phase-1 decoder along with the new case head on the 6000 train cases, with 16 cases per update. The best model on the dev set is saved.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase4/train.py \
    --arch decoder --method full \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/decoder_full


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
6000 train cases, 1988 dev cases
16 cases per update, 375 steps per epoch, 750 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: full, training 111.63M of 111.6M parameters (100.00%)
step 10/750  epoch 1  loss 0.6983  elapsed 0.3 min
step 20/750  epoch 1  loss 0.7068  elapsed 0.6 min
step 30/750  epoch 1  loss 0.7112  elapsed 0.9 min
step 40/750  epoch 1  loss 0.7148  elapsed 1.2 min
step 50/750  epoch 1  loss 0.7087  elapsed 1.5 min
step 60/750  epoch 1  loss 0.6990  elapsed 1.8 min
step 70/750  epoch 1  loss 0.7103  elapsed 2.1 min
step 80/750  epoch 1  loss 0.6884  elapsed 2.4 min
step 90/750  epoch 1  loss 0.7000  elapsed 2.7 min
step 100/750  epoch 1  loss 0.7016  elapsed 3.0 min
step 110/750  epoch 1  loss 0.7057  elapsed 3.3

# 4. Prediction and submissions
## Submission 1: decoder full fine-tuning
Checking the scores for each label on the dev set, then predicting the test cases and making the Codabench submission file.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase4/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/decoder_full

!python /content/drive/MyDrive/CS781/phase4/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p4/decoder_full


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/decoder_full/best_checkpoint.pt (step 750, dev macro-F1 0.6620)
Dev macro-F1: 0.6620
              precision    recall  f1-score   support

           0     0.6364    0.7746    0.6987       994
           1     0.7121    0.5573    0.6253       994

    accuracy                         0.6660      1988
   macro avg     0.6742    0.6660    0.6620      1988
weighted avg     0.6742    0.6660    0.6620      1988

Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/decoder_full/best_checkpoint.pt (step 750, dev macro-F1 0.6620)
1517 test cases, predicted label 1 for 645 of them
Saved /content/drive/MyDrive/CS781/submissions_p4/decoder_full/submission.zip


## LoRMA: convmix
Freezing the Phase-1 convmix model and training only the LoRMA matrices (rank 8) on its attention and convolution layers, along with the case head.



In [1]:
from google.colab import drive
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase4/train.py \
    --arch convmix --method lorma \
    --epochs 3 --learning_rate 1e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/convmix_lorma


Mounted at /content/drive
Using device: cuda
6000 train cases, 1988 dev cases
16 cases per update, 375 steps per epoch, 1125 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: lorma, training 2.59M of 107.5M parameters (2.41%)
step 10/1125  epoch 1  loss 0.6989  elapsed 0.3 min
step 20/1125  epoch 1  loss 0.7202  elapsed 0.5 min
step 30/1125  epoch 1  loss 0.7020  elapsed 0.7 min
step 40/1125  epoch 1  loss 0.7216  elapsed 1.0 min
step 50/1125  epoch 1  loss 0.7054  elapsed 1.2 min
step 60/1125  epoch 1  loss 0.7022  elapsed 1.5 min
step 70/1125  epoch 1  loss 0.7004  elapsed 1.7 min
step 80/1125  epoch 1  loss 0.6953  elapsed 2.0 min
step 90/1125  epoch 1  loss 0.6992  elapsed 2.2 min
step 100/1125  epoch 1  loss 0.7053  elapsed 2.5 min
step 110/1125  epoch 1  loss 0.7066  elapsed 2.8 min
step 120/1125  epoch 1  loss 0.6981  elapsed 3.0 min
  dev macro-F1: 0.5125  (predicted l

## Submission 2: convmix LoRMA
Checking the scores for each label on the dev set, then predicting the test cases and making the Codabench submission file.


In [2]:
!python /content/drive/MyDrive/CS781/phase4/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/convmix_lorma

!python /content/drive/MyDrive/CS781/phase4/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p4/convmix_lorma


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/convmix_lorma/best_checkpoint.pt (step 1125, dev macro-F1 0.5814)
Dev macro-F1: 0.5814
              precision    recall  f1-score   support

           0     0.5972    0.5131    0.5519       994
           1     0.5732    0.6539    0.6109       994

    accuracy                         0.5835      1988
   macro avg     0.5852    0.5835    0.5814      1988
weighted avg     0.5852    0.5835    0.5814      1988

Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/convmix_lorma/best_checkpoint.pt (step 1125, dev macro-F1 0.5814)
1517 test cases, predicted label 1 for 876 of them
Saved /content/drive/MyDrive/CS781/submissions_p4/convmix_lorma/submission.zip


## Partial fine-tuning: encdec
Freezing the lower layers of the Phase-1 encdec encoder and training only its last 3 layers and the case head.


In [3]:
!python /content/drive/MyDrive/CS781/phase4/train.py \
    --arch encdec --method partial --train_top_layers 3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/encdec_partial


Using device: cuda
6000 train cases, 1988 dev cases
16 cases per update, 375 steps per epoch, 750 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 68
Method: partial, training 20.09M of 60.7M parameters (33.08%)
step 10/750  epoch 1  loss 0.7099  elapsed 0.1 min
step 20/750  epoch 1  loss 0.7008  elapsed 0.2 min
step 30/750  epoch 1  loss 0.7102  elapsed 0.3 min
step 40/750  epoch 1  loss 0.7511  elapsed 0.4 min
step 50/750  epoch 1  loss 0.7122  elapsed 0.5 min
step 60/750  epoch 1  loss 0.7225  elapsed 0.6 min
step 70/750  epoch 1  loss 0.6983  elapsed 0.7 min
step 80/750  epoch 1  loss 0.6853  elapsed 0.8 min
step 90/750  epoch 1  loss 0.7213  elapsed 0.8 min
step 100/750  epoch 1  loss 0.7202  elapsed 0.9 min
step 110/750  epoch 1  loss 0.7130  elapsed 1.0 min
step 120/750  epoch 1  loss 0.7137  elapsed 1.1 min
  dev macro-F1: 0.3333  (predicted label 1 for 0 of 1988 dev cases)
step

## Submission 3: encdec partial fine-tuning
Checking the scores for each label on the dev set, then predicting the test cases and making the Codabench submission file.


In [4]:
!python /content/drive/MyDrive/CS781/phase4/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/encdec_partial/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/encdec_partial

!python /content/drive/MyDrive/CS781/phase4/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/encdec_partial/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p4/encdec_partial


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/encdec_partial/best_checkpoint.pt (step 625, dev macro-F1 0.5013)
Dev macro-F1: 0.5013
              precision    recall  f1-score   support

           0     0.5329    0.3260    0.4045       994
           1     0.5145    0.7143    0.5981       994

    accuracy                         0.5201      1988
   macro avg     0.5237    0.5201    0.5013      1988
weighted avg     0.5237    0.5201    0.5013      1988

Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/encdec_partial/best_checkpoint.pt (step 625, dev macro-F1 0.5013)
1517 test cases, predicted label 1 for 1066 of them
Saved /content/drive/MyDrive/CS781/submissions_p4/encdec_partial/submission.zip


## Knowledge distillation: decoder
Training a smaller 6-layer decoder, taken from the first 6 layers of the Phase-1 decoder, to learn from the trained decoder_full model and from the gold labels.


In [7]:
!python /content/drive/MyDrive/CS781/phase4/distill.py \
    --teacher_ckpt /content/drive/MyDrive/CS781/ckpt_p4/decoder_full/best_checkpoint.pt \
    --p1_ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/decoder_distill


Using device: cuda
Student backbone loaded, missing keys: 0, unused keys: 43
Teacher: 111.6M parameters, student: 69.2M parameters (62% of the teacher)
16 cases per update, 375 steps per epoch, 750 steps total
step 10/750  epoch 1  loss 0.4551  elapsed 0.2 min
step 20/750  epoch 1  loss 0.4551  elapsed 0.5 min
step 30/750  epoch 1  loss 0.4821  elapsed 0.7 min
step 40/750  epoch 1  loss 0.4662  elapsed 1.0 min
step 50/750  epoch 1  loss 0.4787  elapsed 1.2 min
step 60/750  epoch 1  loss 0.4797  elapsed 1.5 min
step 70/750  epoch 1  loss 0.4639  elapsed 1.7 min
step 80/750  epoch 1  loss 0.4443  elapsed 2.0 min
step 90/750  epoch 1  loss 0.4586  elapsed 2.2 min
step 100/750  epoch 1  loss 0.4098  elapsed 2.5 min
step 110/750  epoch 1  loss 0.4810  elapsed 2.7 min
step 120/750  epoch 1  loss 0.4382  elapsed 3.0 min
  dev macro-F1: 0.5651  (predicted label 1 for 942 of 1988 dev cases)
step 130/750  epoch 1  loss 0.4666  elapsed 4.0 min
step 140/750  epoch 1  loss 0.4435  elapsed 4.2 min
s

## Submission 4: decoder knowledge distillation
Checking the 6-layer student on the dev set and making its test submission.


In [4]:
!python /content/drive/MyDrive/CS781/phase4/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/decoder_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p4/decoder_distill

!python /content/drive/MyDrive/CS781/phase4/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p4/decoder_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p4 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p4/decoder_distill


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/decoder_distill/best_checkpoint.pt (step 500, dev macro-F1 0.6257)
Dev macro-F1: 0.6257
              precision    recall  f1-score   support

           0     0.6501    0.5533    0.5978       994
           1     0.6112    0.7022    0.6536       994

    accuracy                         0.6278      1988
   macro avg     0.6307    0.6278    0.6257      1988
weighted avg     0.6307    0.6278    0.6257      1988

Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p4/decoder_distill/best_checkpoint.pt (step 500, dev macro-F1 0.6257)
1517 test cases, predicted label 1 for 855 of them
Saved /content/drive/MyDrive/CS781/submissions_p4/decoder_distill/submission.zip
